# 🧠 T4CT — Two-Photon Ca²⁺ Imaging Pipeline (Team 8)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leonor-sinogas/T4CT-team-8/blob/main/notebooks/T4CT_demo.ipynb)

**Denoise → motion-correct → segment neuron footprints → extract Ca²⁺ traces** on a two-photon recording (10,000 frames @ 30 fps), using **suite2p** on a Colab **T4**.

1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all`
3. On the **first** run, the install cell restarts the runtime (suite2p needs a fresh numpy/numba). When it comes back, just **Run all again** — it skips the install and runs through.

Works end-to-end on a **synthetic movie** today; swap in the real ~5 GB TIFF in the *Load the data* cell on workshop day. Team workflow notes are at the bottom.

In [ ]:
# 1) Check the GPU — should list a Tesla T4
!nvidia-smi -L || echo 'No GPU — Runtime > Change runtime type > T4 GPU'

In [ ]:
# 2) Pull the team's code from GitHub (works in Colab; no-op when run locally)
import os, sys
if os.path.isdir('/content'):
    REPO_URL = 'https://github.com/leonor-sinogas/T4CT-team-8.git'
    REPO_DIR = '/content/T4CT-team-8'
    if os.path.isdir(REPO_DIR):
        !cd {REPO_DIR} && git pull --ff-only
    else:
        !git clone {REPO_URL} {REPO_DIR}
else:
    REPO_DIR = os.path.abspath('..')   # running locally from notebooks/
os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'src'))
print('repo:', REPO_DIR)

In [ ]:
# 3) Install suite2p (+ tifffile). torch etc. are already on Colab.
#    suite2p pins a numpy/numba combo, so a ONE-TIME runtime restart is needed
#    after the first install. This cell installs, then restarts automatically;
#    when the runtime comes back just Run all again and it skips ahead.
import importlib.util, os, subprocess, sys

if importlib.util.find_spec('suite2p') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
    print('\u2705 Installed suite2p. Restarting runtime \u2014 re-run from the top (Run all) when it returns.')
    os.kill(os.getpid(), 9)   # force restart so the new numpy/numba load cleanly
else:
    import suite2p
    print('suite2p', getattr(suite2p, 'version', '?'), 'already installed \u2014 skipping')

In [ ]:
# 4) Verify the environment (after the restart this should all pass)
import suite2p, torch, numpy, numba
from suite2p import run_s2p   # the headless pipeline API we use (suite2p 1.x, no GUI)
print('suite2p :', getattr(suite2p, 'version', '?'))
print('torch   :', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
print('numpy   :', numpy.__version__, '| numba:', numba.__version__)
print('✅ suite2p pipeline ready')

## Load the data
Synthetic movie by default (no download). On workshop day, uncomment **Option A** and point it at the real TIFF in your Drive.

In [ ]:
# Get a movie of shape (T, H, W)
from t4ct import data, motion, denoise, segment, traces, viz
import numpy as np
FPS = 30.0

# --- Option A: your recording on Google Drive (uncomment + set the path) ---
# from google.colab import drive; drive.mount('/content/drive')
# TIFF_PATH = '/content/drive/MyDrive/YOUR_FOLDER/your_recording.tif'   # <-- set this
# mov = data.load_tiff(TIFF_PATH, memmap=True)        # large files -> memmap, not RAM
# true_fp = true_tr = None

# --- Option B: synthetic movie (default; runs out of the box) ---
mov, true_fp, true_tr = data.synthetic_movie(n_frames=600, size=256,
                                             n_cells=80, motion=True, seed=0)
print('movie:', mov.shape, mov.dtype)

## 1 · Inspect the recording

In [ ]:
viz.show_summary(mov)    # mean / max / correlation image
viz.montage(mov, n=8)    # a few frames across time

## 2 · Motion correction
Lightweight rigid baseline (fine for short clips). For the full 10,000-frame recording, suite2p's built-in registration (step 4) is faster and supports non-rigid correction.

In [ ]:
mov_mc, shifts = motion.register_rigid(mov)
import matplotlib.pyplot as plt
plt.plot(shifts); plt.legend(['dy', 'dx']); plt.xlabel('frame'); plt.ylabel('px')
plt.title('estimated motion'); plt.show()
viz.show_summary(mov_mc)

## 3 · Denoising (intermediate milestone)
Classical low-rank (PCA) baseline — cleans the movie and sharpens the correlation image. The advanced milestone swaps this for DeepCAD-RT / DeepInterpolation (see the bottom).

In [ ]:
mov_dn = denoise.pca_denoise(mov_mc, n_components=50)
viz.show_summary(mov_dn)
if true_fp is not None:
    print('PSNR raw-mean vs denoised-mean:',
          round(denoise.psnr(data.mean_image(mov), data.mean_image(mov_dn)), 2), 'dB')

## 4 · Segmentation + traces — the full suite2p pipeline (minimum milestone)
suite2p does **its own motion correction**, ROI detection, deconvolution and trace extraction, so we feed it the **raw** movie. `tau` is the GCaMP decay time (s). On the 10,000-frame recording this is the slow step (minutes on the T4).

In [ ]:
out = segment.run_suite2p(mov, save_path='/content/s2p_out', fs=FPS, tau=1.0)
cells = out['iscell'][:, 0].astype(bool)
print('suite2p ROIs classified as cells:', int(cells.sum()), '/', len(out['stat']))

viz.plot_footprints(out['footprints'][cells], out['ops']['meanImg'], title='suite2p ROIs')
F_corr = traces.neuropil_correct(out['F'][cells], out['Fneu'][cells])
viz.plot_traces(traces.dff(F_corr), fps=FPS, n=50)

### Baseline alternative (no suite2p)
Correlation-image segmentation — handy for a quick look or a sanity check against suite2p.

In [ ]:
# Heads-up: the correlation image is computed over ALL frames, so on a full
# high-res recording this baseline can exhaust RAM (the kernel may crash).
# Subsample first if needed, e.g. correlation_segment(mov_dn[::2], ...) or crop the FOV.
fps_base, ci = segment.correlation_segment(mov_dn, min_corr=0.3)
F_base = traces.extract_traces(mov_dn, fps_base)
viz.plot_footprints(fps_base, ci, title='correlation-image ROIs')
viz.plot_traces(traces.dff(F_base), fps=FPS, n=30)
print('baseline ROIs:', len(fps_base))

## 5 · Advanced — deep self-supervised denoising (uses the T4)
Both methods train on the noisy movie itself (no clean targets needed):

- **DeepCAD-RT** — `!pip install deepcad` · https://github.com/cabooster/DeepCAD-RT
- **DeepInterpolation** — https://github.com/AllenInstitute/deepinterpolation

`denoise.deepcad_denoise(mov)` is a stub to wire up. The advanced milestone is to fold in priors — Ca²⁺ dynamics statistics and the imaging PSF — then re-run step 4 and compare ROI counts / trace SNR.

## 6 · Next steps
1. Mount Drive and load the real TIFF (Load-the-data cell, Option A).
2. Tune suite2p `ops` for this data (`tau`, `diameter`/`anatomical_only`, thresholds).
3. Compare segmentation **with vs without** denoising/motion-correction (milestone delta).
4. Quantify: # neurons, ΔF/F SNR, event rates; export traces (`np.save`).
5. Attempt the advanced DL denoiser and re-measure.

---
## 👥 Collaboration
**Code** lives in this GitHub repo — each person runs their own runtime:
- Cell 2 runs `git pull` so you start on the latest code each session.
- Edit in `src/t4ct/`, commit on a branch / open a PR. Keep the notebook thin.
- Commit small fixes from Colab: `!git add -A && git commit -m '...' && git push` (push needs a GitHub Personal Access Token).

**Real-time co-editing of the notebook** (Google-Docs style):
- `File → Save a copy in Drive`, then **Share** that copy — you can all edit/run it live.
- Fold changes back with `File → Save a copy in GitHub`.

**Big files (the 5 GB recording):** Google Drive (Load-the-data cell), never git.